# learned-copper: UNet at 256 px on a free cloud GPU

Runs `scripts/train_unet.py` from [github.com/Om-singhaI/learned-copper](https://github.com/Om-singhaI/learned-copper) on Kaggle (P100 or T4) or on Colab (T4). The rasters arrive as the bundle written by `scripts/make_raster_bundle.py`: attach it as a private Kaggle dataset, or place it in the Drive folder named in the settings cell for Colab. The steps, the time budget and how to resume are in `docs/cloud-training.md`.

Keep this notebook and the dataset private. The rasters derive from PCBench boards and must not be redistributed.

In [ ]:
# Settings. The platform is detected from the file system; set PLATFORM by hand to force one.
import os
import time

T0 = time.time()
PLATFORM = "kaggle" if os.path.isdir("/kaggle") else "colab"
REPO_URL = "https://github.com/Om-singhaI/learned-copper"
BUNDLE_GLOB = "learned-copper-rasters-*.tar.gz"
KAGGLE_DATASET = ""        # slug of the attached dataset under /kaggle/input; empty searches every input
DRIVE_DIR = "/content/drive/MyDrive/learned-copper"   # Colab: the bundle sits here and models are written here
SIZE = 256
EPOCHS = 30
WORKERS = 2
RUN_OWNER = True           # owner split, the primary protocol
RUN_RANDOM = False         # random split; give it its own session so each split gets the whole budget
TIME_LIMIT_MINUTES = 660 if PLATFORM == "kaggle" else 170   # training wall time before a clean stop
CLEAN_DATA_AT_END = True   # Kaggle: remove the extracted rasters so they are not saved with the output

if PLATFORM == "kaggle":
    WORK, MODELS = "/kaggle/working", "/kaggle/working/models"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK, MODELS = "/content", f"{DRIVE_DIR}/models"
REPO = f"{WORK}/learned-copper"
os.makedirs(MODELS, exist_ok=True)
print(f"platform {PLATFORM}, repo {REPO}, models {MODELS}, time limit {TIME_LIMIT_MINUTES} min")

In [ ]:
# Clone the repository at main, or update a clone left by an earlier run of this cell.
import subprocess

if os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main", "--depth", "1", REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Install whatever pyproject.toml lists that this image lacks. torch is preinstalled on Kaggle and on
# Colab and is left alone, so the CUDA build stays the one the image was tested with.
import importlib
import re
import sys
import tomllib

with open(f"{REPO}/pyproject.toml", "rb") as f:
    deps = tomllib.load(f)["project"]["dependencies"]
IMPORT_NAMES = {"scikit-learn": "sklearn"}
missing = []
for dep in deps:
    name = re.split(r"[<>=!~\[ ;]", dep)[0]
    if name == "torch":
        continue
    try:
        importlib.import_module(IMPORT_NAMES.get(name, name.replace("-", "_")))
    except ImportError:
        missing.append(name)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
print("installed:", missing or "nothing, every package was present")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU: enable the accelerator)")

In [ ]:
# Find the raster bundle and unpack it into the repository's data/ and splits/.
import glob
import hashlib
import json
import shutil
import tarfile

if PLATFORM == "kaggle":
    roots = [f"/kaggle/input/{KAGGLE_DATASET}" if KAGGLE_DATASET else "/kaggle/input"]
else:
    roots = [DRIVE_DIR]
bundles = sorted(p for r in roots for p in glob.glob(f"{r}/**/{BUNDLE_GLOB}", recursive=True))
raster_dir, splits_dir = f"{REPO}/data/raster", f"{REPO}/splits"
if bundles:
    bundle = bundles[-1]
    h = hashlib.sha256()
    with open(bundle, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    print(f"{bundle}: {os.path.getsize(bundle) / 1e6:.1f} MB, sha256 {h.hexdigest()}")
    with tarfile.open(bundle) as tar:
        try:
            tar.extractall(REPO, filter="data")
        except TypeError:  # an older tarfile without the filter argument
            tar.extractall(REPO)
else:
    # Kaggle unpacks some archives on upload; look for the extracted tree instead.
    found = sorted(p for r in roots for p in glob.glob(f"{r}/**/splits/raster_manifest.json", recursive=True))
    if not found:
        raise SystemExit(f"no bundle found under {roots}; attach the dataset or check BUNDLE_GLOB")
    src = os.path.dirname(os.path.dirname(found[-1]))
    shutil.copytree(f"{src}/data/raster", raster_dir, dirs_exist_ok=True)
    for name in ("raster_manifest.json", "mirror_pool.json"):
        shutil.copy(f"{src}/splits/{name}", splits_dir)
    print(f"copied the unpacked bundle from {src}")
n_files = len(glob.glob(f"{raster_dir}/*.npz"))
with open(f"{splits_dir}/raster_manifest.json") as f:
    boards = json.load(f)["boards"]
print(f"{n_files} raster files, {len(boards)} boards in the manifest, {sum(b['held_out'] for b in boards)} held out")
assert n_files == len(boards), "raster files and manifest disagree"

In [ ]:
# One training run per split. On Kaggle a previous version's output can be attached as an input;
# its last.pt is copied in so --resume continues from it. On Colab the Drive folder persists by itself.
# The time limit handed to train_unet.py is what remains of TIME_LIMIT_MINUTES, shared between the
# runs still to come, so the script stops cleanly with last.pt and metrics.json written.
def run_split(split, runs_left):
    out = f"{MODELS}/unet_{split}_{SIZE}"
    os.makedirs(out, exist_ok=True)
    if PLATFORM == "kaggle" and not os.path.exists(f"{out}/last.pt"):
        previous = sorted(glob.glob(f"/kaggle/input/**/unet_{split}_{SIZE}/last.pt", recursive=True))
        if previous:
            shutil.copytree(os.path.dirname(previous[-1]), out, dirs_exist_ok=True)
            print("copied the previous output from", os.path.dirname(previous[-1]))
    minutes_left = TIME_LIMIT_MINUTES - (time.time() - T0) / 60
    budget = max(1, int(minutes_left / runs_left))
    cmd = [sys.executable, "scripts/train_unet.py", "--split", split, "--size", str(SIZE),
           "--epochs", str(EPOCHS), "--eval-held-out", "--amp", "--workers", str(WORKERS),
           "--time-limit-minutes", str(budget), "--resume", "--out", out]
    print(" ".join(cmd), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    with subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=env) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    if proc.returncode:
        raise RuntimeError(f"train_unet.py exited with code {proc.returncode}")
    return out

In [ ]:
# Owner split at 256 px.
if RUN_OWNER:
    run_split("owner", runs_left=1 + int(RUN_RANDOM))

In [ ]:
# Random split at 256 px, the leakage comparison.
if RUN_RANDOM:
    run_split("random", runs_left=1)

In [ ]:
# Summarise the runs and zip the output directories. On Kaggle both the zip and models/ appear under
# Output once the version finishes; on Colab the zip is written next to the models on Drive.
def summary(d):
    name = os.path.basename(d)
    if not os.path.exists(f"{d}/metrics.json"):
        return f"{name}: no metrics.json yet"
    with open(f"{d}/metrics.json") as f:
        m = json.load(f)
    line = (f"{name}: {m.get('status')}, {len(m['epochs'])} of {m['args']['epochs']} epochs done, "
            f"best epoch {m['best_epoch']}")
    if m["epochs"]:
        line += f", last epoch {m['epochs'][-1]['seconds'] / 60:.1f} min"
    if m.get("final_val"):
        line += f", val IoU {m['final_val']['overall']['iou']:.3f}"
    if os.path.exists(f"{d}/held_out_metrics.json"):
        with open(f"{d}/held_out_metrics.json") as f:
            line += f", held out IoU {json.load(f)['unet']['overall']['iou']:.3f}"
    return line


for d in sorted(glob.glob(f"{MODELS}/unet_*")):
    print(summary(d))
zip_dir = WORK if PLATFORM == "kaggle" else DRIVE_DIR
archive = shutil.make_archive(f"{zip_dir}/unet_{SIZE}_outputs", "zip", MODELS)
print(f"{archive}: {os.path.getsize(archive) / 1e6:.1f} MB")
if CLEAN_DATA_AT_END and PLATFORM == "kaggle":
    shutil.rmtree(raster_dir, ignore_errors=True)
    print("removed the extracted rasters from the working directory")